# Project Finance Analysis: Rail Development (Integrated NPV & Societal IRR)

This analysis integrates monetized societal impacts and nature dependencies into traditional financial metrics. It aligns with the framework of **Integrated NPV (iNPV)**, **Societal IRR (S-IRR)**, and **Risk-Adjusted WACC**.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 1. PARAMETERS & INPUTS
# ══════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import numpy_financial as npf
from plotly.subplots import make_subplots

# Financial Constants
CAPEX_FINANCIAL = 1_998_000_000  # From Rail_EU_DEV investment_usd
ANNUAL_CASH_FLOW = 150_000_000   # Assumptions for rail operations
WACC_BASE = 0.07                 # 7% base discount rate
PROJECT_LIFETIME = 20            # Years
BOND_LIFETIME = 10               # Years

# Monetization Coefficients (WifOR / Nordhaus Estimates)
GHG_COST_PER_T = 200             # USD / tCO2e
WATER_COST_PER_M3 = 0.50         # USD / m3
EMPLOYMENT_VALUE_PER_FTE = 1760 * 15 # USD / FTE (1760 hrs * 15 $/hr value-add)

# Construction Impacts (Nature Debt - Rail_EU_DEV)
CONSTRUCTION_GHG_T = 787_208
CONSTRUCTION_WATER_M3 = 5_336_459
S_CONSTRUCTION = (CONSTRUCTION_GHG_T * GHG_COST_PER_T) + (CONSTRUCTION_WATER_M3 * WATER_COST_PER_M3)

# Annual Operations Impacts (Nature Dividend - Scaled Rail_EU_OP1)
# We scale OP1 (which is small) to match a full-scale operation relative to CAPEX
OP_GHG_SAVINGS_T = 50_000        # Assumed avoided emissions from shifting road to rail
OP_EMPLOYMENT_FTE = 1_500        # Direct and indirect operational employment
S_OPERATIONS = (OP_EMPLOYMENT_FTE * EMPLOYMENT_VALUE_PER_FTE) - (2000 * GHG_COST_PER_T) # 2000t operational footprint

# Nature Dependency (Section 3)
WWF_WATER_RISK = 3.378           # From dependency_summary.csv (wrf_composite)
DEP_FACTOR = 0.85                # Resiliency dampening factor
SUSTAINABILITY_DISCOUNT = 0.0010 # 10 basis points


In [ ]:
# ══════════════════════════════════════════════════════════════════
# 2. INTEGRATED NPV (iNPV) CALCULATION
# ══════════════════════════════════════════════════════════════════

# Cash Flow Arrays
years = np.arange(PROJECT_LIFETIME + 1)
cf_financial = np.full(PROJECT_LIFETIME + 1, ANNUAL_CASH_FLOW)
cf_financial[0] = -CAPEX_FINANCIAL

cf_integrated = np.full(PROJECT_LIFETIME + 1, ANNUAL_CASH_FLOW + S_OPERATIONS)
cf_integrated[0] = -(CAPEX_FINANCIAL + S_CONSTRUCTION)

# NPV Calculations
npv_standard = npf.npv(WACC_BASE, cf_financial)
npv_integrated = npf.npv(WACC_BASE, cf_integrated)

print(f"Standard NPV:   ${npv_standard:,.2f}")
print(f"Integrated NPV: ${npv_integrated:,.2f}")
print(f"Value Delta:    ${(npv_integrated - npv_standard):,.2f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# 3. SOCIETAL IRR (S-IRR)
# ══════════════════════════════════════════════════════════════════

irr_standard = npf.irr(cf_financial)
irr_societal = npf.irr(cf_integrated)

print(f"Financial IRR: {irr_standard:.2%}")
print(f"Societal IRR:  {irr_societal:.2%}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# 4. RISK-ADJUSTED WACC (ESG ALPHA/BETA)
# ══════════════════════════════════════════════════════════════════

# Logic: If DEP_FACTOR < 1.0 and WATER_RISK is manageable, apply discount
wacc_adjusted = WACC_BASE
if DEP_FACTOR < 1.0 and WWF_WATER_RISK < 4.0:
    wacc_adjusted -= SUSTAINABILITY_DISCOUNT

print(f"Base WACC:     {WACC_BASE:.2%}")
print(f"Adjusted WACC: {wacc_adjusted:.2%}")

# Re-calculate Integrated NPV with Adjusted WACC
npv_integrated_adj = npf.npv(wacc_adjusted, cf_integrated)
print(f"iNPV (WACC Adj): ${npv_integrated_adj:,.2f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# 5. VISUALIZATION
# ══════════════════════════════════════════════════════════════════

fig = make_subplots(rows=1, cols=2, subplot_types=['bar', 'indicator'])

# NPV Comparison
fig.add_trace(go.Bar(
    x=['Standard NPV', 'Integrated NPV'],
    y=[npv_standard, npv_integrated],
    marker_color=['#4e79a7', '#59a14f'],
    name="NPV comparison"
), row=1, col=1)

# S-IRR Gauge
fig.add_trace(go.Indicator(
    mode = "gauge+number",
    value = irr_societal * 100,
    title = {'text': "Societal IRR (%)"},
    gauge = {
        'axis': {'range': [None, 20]},
        'steps': [
            {'range': [0, irr_standard*100], 'color': "lightgray"},
            {'range': [irr_standard*100, 20], 'color': "lightgreen"}
        ],
        'threshold': {
            'line': {'color': "red", 'width': 4},
            'thickness': 0.75,
            'value': WACC_BASE * 100
        }
    }
), row=1, col=2)

fig.update_layout(title_text="Investment Analysis: Financial vs. Societal Metrics", template="plotly_white")
fig.show()
